# Probability Interview Questions & Answers for Data Scientists

Companion to the [complete article](https://todatabeyond.com/blog/probability-interview-questions-and-answers-for-data-scientists). Run all cells in order on Python 3.11+ in Jupyter/Colab. Setup actually installs NumPy 2.3.3 into the current interpreter. Everything else is standard library; no API keys, hosted services, GPU or datasets. Only package installation needs network access. The complete original referenced Monte Carlo gist is in the nonexecuting appendix, with its retrieved revision.

All twelve questions receive reproducible computational examples. Fraction arithmetic supplies exact discrete results; simulations are seeded empirical checks, not proofs. Q4 retains independent uniform draws and np.median, batching the original million trials to bound memory and adding count validation and a seed. No source figure is replaced by a generated diagram.

Original explanatory prose is preserved in the blog. Review findings stay here and in the private import review: Q2 has a multiplication/exponent typo; Q7 prints 356p30 where its original figure correctly uses 365p30 and truncates the decimal. Q6's six-toss support is 0 through 6; 0,1,2 defines a cumulative event, not the whole support. Exact code uses consistent formulas.

Q8 question does not specify the unfair bias, but its source solution assumes an always-tails coin. Make the tail probability and prior explicit parameters: q=1 gives 32/33, while q=4/5 gives a different answer. The evidence denominator needs parentheses. No unique numeric posterior follows from the word unfair alone.

Q9 uses two independent uniform breakpoints, which induces the uniform simplex model in the source diagram; alternative ways to break a stick need not have the same answer. All three lengths must be below one half; source wording no two lengths is not sufficient. Q10 assumes independently uniform boundary endpoints or equivalently an equiprobable pairing of four IID endpoints; random chords has other sampling models. Cyclic pairing (1,3)/(2,4) crosses; point labels/order in the source image differ from the prose list. The code fixes one explicit cyclic order.

Q11 assumes six independent intervals with equal no-event probability. The five-minute marginal alone does not establish the half-hour result without dependence assumptions. Exact calculation is 0.622850484375; the source decimal is truncated. Birthdays use IID uniform 365-day years, not real population seasonality or leap days.

Von Neumann extraction requires independent tosses with the same fixed bias and 0<p<1. The extractor need not know p, but a deterministic coin cannot produce randomness. A finite trial budget raises TimeoutError rather than silently returning a biased default. Exact accepted-pair symmetry establishes fairness under the stated model; a single sample count is only a smoke check. Examples demonstrate default synthetic models, not measured real-world aircraft or birth data. Committed notebook counts are null and outputs empty.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--disable-pip-version-check', 'numpy==2.3.3'], check=True)

from fractions import Fraction
from itertools import product, combinations
from math import comb, perm, sqrt
import numpy as np

def require_count(value, name, minimum=0):
    if not isinstance(value, int) or isinstance(value, bool) or value < minimum:
        raise ValueError(f'{name} must be an integer >= {minimum}.')

print('Python:', sys.version.split()[0], '| NumPy:', np.__version__)
print('Exact rational calculations and seeded simulations; no API keys, external data or GPU.')


## 1. Competing Coin Sequences


In [ ]:
first_two = list(product('HT', repeat=2))
coin_win = Fraction(sum(pair == ('H', 'H') for pair in first_two), len(first_two))
print('First-two-toss outcomes:', [''.join(pair) for pair in first_two])
print('Win probability:', coin_win)
print('Friend eventually wins:', 1 - coin_win)


## 2. Consecutive Threes in Dice Rolls


In [ ]:
rolls = list(product(range(1, 7), repeat=3))
consecutive = [r for r in rolls if r[:2] == (3, 3) or r[1:] == (3, 3)]
dice_probability = Fraction(len(consecutive), len(rolls))
inclusion_exclusion = 2 * Fraction(1, 6) ** 2 - Fraction(1, 6) ** 3
print('Favourable outcomes:', len(consecutive), 'of', len(rolls))
print('Enumeration:', dice_probability)
print('Inclusion-exclusion:', inclusion_exclusion)


## 3. A Dice Sum Divisible by Six


In [ ]:
def residue_counts(n_dice, sides=6):
    require_count(n_dice, 'n_dice')
    require_count(sides, 'sides', minimum=1)
    counts = [1] + [0] * (sides - 1)
    for _ in range(n_dice):
        following = [0] * sides
        for residue, count in enumerate(counts):
            for face in range(1, sides + 1):
                following[(residue + face) % sides] += count
        counts = following
    return counts

ten_dice_counts = residue_counts(10)
divisible_probability = Fraction(ten_dice_counts[0], 6 ** 10)
print('Counts for residues 0–5:', ten_dice_counts)
print('Probability divisible by six:', divisible_probability)


## 4. The Median of Three Uniform Draws


In [ ]:
def monte_carlo(n, seed=42, batch_size=100_000):
    require_count(n, 'n', minimum=1)
    require_count(batch_size, 'batch_size', minimum=1)
    rng = np.random.default_rng(seed)
    count = 0
    for start in range(0, n, batch_size):
        size = min(batch_size, n - start)
        draws = rng.uniform(0, 2, size=(size, 3))
        count += int(np.count_nonzero(np.median(draws, axis=1) > 1.5))
    return count / n

upper_tail = Fraction(1, 4)
median_exact = comb(3, 2) * upper_tail ** 2 * (1 - upper_tail) + upper_tail ** 3
median_trials = 1_000_000
median_estimate = monte_carlo(median_trials)
print('Exact probability:', median_exact, '=', float(median_exact))
print('Monte Carlo trials:', median_trials)
print(f'Seeded estimate: {median_estimate:.6f}')
print(f'Absolute error: {abs(median_estimate - float(median_exact)):.6f}')


## 5. A Card and Its Double


In [ ]:
card_pairs = list(combinations(range(1, 101), 2))
doubled_pairs = [(left, right) for left, right in card_pairs if right == 2 * left]
double_probability = Fraction(len(doubled_pairs), len(card_pairs))
print('Favourable unordered pairs:', len(doubled_pairs))
print('All unordered pairs:', len(card_pairs))
print('Probability:', double_probability, '=', float(double_probability))


## 6. Bernoulli and Binomial Distributions


In [ ]:
def binomial_pmf(n, success_probability):
    require_count(n, 'n')
    p = Fraction(success_probability)
    if not 0 <= p <= 1:
        raise ValueError('Success probability must be in [0, 1].')
    return [comb(n, k) * p ** k * (1 - p) ** (n - k) for k in range(n + 1)]

p = Fraction(1, 2)
bernoulli = binomial_pmf(1, p)
binomial = binomial_pmf(6, p)
print('Bernoulli probabilities for 0, 1:', [str(value) for value in bernoulli])
print('Binomial support:', list(range(7)))
print('Binomial probabilities:', [str(value) for value in binomial])
print('P(X <= 2) for six fair tosses:', sum(binomial[:3]))
print('Bernoulli mean/variance:', p, p * (1 - p))
print('Binomial mean/variance:', 6 * p, 6 * p * (1 - p))


## 7. The Birthday Problem


In [ ]:
def distinct_birthdays(people, days=365):
    """Independent uniform birthdays over days equally likely dates."""
    require_count(people, 'people')
    require_count(days, 'days', minimum=1)
    if people > days:
        return Fraction(0)
    return Fraction(perm(days, people), days ** people)

birthday_distinct = distinct_birthdays(30)
print(f'All thirty distinct: {float(birthday_distinct):.10f}')
print(f'At least one match, 23 people: {float(1 - distinct_birthdays(23)):.10f}')
print(f'At least one match, 57 people: {float(1 - distinct_birthdays(57)):.10f}')


## 8. Bayesian Coin Identification


In [ ]:
def posterior_unfair(tails, unfair_tail_probability, prior=Fraction(1, 2)):
    """IID flips; supply the unfair coin's tail probability and prior explicitly."""
    require_count(tails, 'tails')
    q, prior = Fraction(unfair_tail_probability), Fraction(prior)
    if not 0 <= q <= 1 or not 0 <= prior <= 1:
        raise ValueError('Probabilities must be in [0, 1].')
    unfair_weight = prior * q ** tails
    fair_weight = (1 - prior) * Fraction(1, 2) ** tails
    if unfair_weight + fair_weight == 0:
        raise ValueError('The observed event has zero probability.')
    return unfair_weight / (unfair_weight + fair_weight)

always_tails_posterior = posterior_unfair(5, Fraction(1))
other_bias_posterior = posterior_unfair(5, Fraction(4, 5))
print('Unfair coin tails probability = 1:', always_tails_posterior)
print(f'Posterior: {float(always_tails_posterior):.6%}')
print('Unfair coin tails probability = 4/5:', other_bias_posterior)
print(f'Posterior: {float(other_bias_posterior):.6%}')


## 9. The Broken-Stick Triangle


In [ ]:
def broken_stick_trials(n, seed=43):
    """Use two independent uniform cut positions on the unit stick."""
    require_count(n, 'n', minimum=1)
    cuts = np.sort(np.random.default_rng(seed).uniform(0, 1, size=(n, 2)), axis=1)
    pieces = np.column_stack((cuts[:, 0], cuts[:, 1] - cuts[:, 0], 1 - cuts[:, 1]))
    successes = np.max(pieces, axis=1) < 0.5
    return pieces, successes

stick_exact = Fraction(1, 8) / Fraction(1, 2)
stick_pieces, stick_successes = broken_stick_trials(100_000)
print('Central region / simplex area:', stick_exact)
print('Simulated cut pairs:', len(stick_successes))
print(f'Seeded triangle estimate: {stick_successes.mean():.6f}')


## 10. Intersecting Random Chords


In [ ]:
def chords_cross(first, second):
    """Four distinct endpoints numbered in cyclic order around the circle."""
    if len(set(first + second)) != 4:
        raise ValueError('Four distinct endpoints are required.')
    left, right = sorted(first)
    return (left < second[0] < right) != (left < second[1] < right)

pairings = [((1, 2), (3, 4)), ((1, 3), (2, 4)), ((1, 4), (2, 3))]
crossing_flags = [chords_cross(first, second) for first, second in pairings]
chord_exact = Fraction(sum(crossing_flags), len(pairings))
# Each chord uses two independent uniform endpoints on the circumference.
endpoints = np.random.default_rng(44).uniform(0, 1, size=(100_000, 4))
lower = np.minimum(endpoints[:, 0], endpoints[:, 1])
upper = np.maximum(endpoints[:, 0], endpoints[:, 1])
inside_a = (lower < endpoints[:, 2]) & (endpoints[:, 2] < upper)
inside_b = (lower < endpoints[:, 3]) & (endpoints[:, 3] < upper)
chord_successes = inside_a != inside_b
print('Cyclic pairing intersections:', crossing_flags)
print('Exact probability:', chord_exact)
print(f'Seeded endpoint estimate: {chord_successes.mean():.6f}')


## 11. Airplanes over Independent Intervals


In [ ]:
def at_least_one_independent(probability_per_interval, intervals):
    require_count(intervals, 'intervals')
    p = Fraction(probability_per_interval)
    if not 0 <= p <= 1:
        raise ValueError('Probability must be in [0, 1].')
    return 1 - (1 - p) ** intervals

airplane_probability = at_least_one_independent(Fraction(3, 20), 6)
print('Six independent five-minute intervals:', airplane_probability)
print(f'Probability: {float(airplane_probability):.12f}')


## 12. Fair Odds from a Biased Coin


In [ ]:
def fair_outcome(toss, max_pairs=10_000):
    """IID fixed-bias H/T tosses; keep HT as H and TH as T."""
    require_count(max_pairs, 'max_pairs', minimum=1)
    for _ in range(max_pairs):
        first, second = toss(), toss()
        if first not in ('H', 'T') or second not in ('H', 'T'):
            raise ValueError('toss must return H or T.')
        if first != second:
            return first
    raise TimeoutError('No unequal pair within the trial budget.')

biased_rng = np.random.default_rng(45)
raw_tosses = [0]
def toss_unknown_coin():
    raw_tosses[0] += 1
    return 'H' if biased_rng.random() < 0.8 else 'T'

fair_samples = [fair_outcome(toss_unknown_coin) for _ in range(10_000)]
p_head = Fraction(4, 5)
p_ht = p_head * (1 - p_head)
p_th = (1 - p_head) * p_head
print('Exact accepted-pair H probability:', p_ht / (p_ht + p_th))
print('Extracted outcomes:', len(fair_samples), '| Raw tosses:', raw_tosses[0])
print('Observed H/T counts:', fair_samples.count('H'), fair_samples.count('T'))


## Independent regression checks


In [ ]:
from collections import defaultdict
import math

def raises(error, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except error:
        return
    raise AssertionError('Expected ' + error.__name__)

# Independent finite-horizon enumeration checks first-winning pattern semantics.
for length in range(2, 11):
    winners = {'HH': 0, 'TH': 0, None: 0}
    for tosses in product('HT', repeat=length):
        winner = next((''.join(tosses[i:i + 2]) for i in range(length - 1)
                       if ''.join(tosses[i:i + 2]) in ('HH', 'TH')), None)
        winners[winner] += 1
    assert Fraction(winners['HH'], 2 ** length) == Fraction(1, 4)
    assert winners[None] == 2  # Remaining mass tends to zero.
assert coin_win == Fraction(1, 4)
assert len(set(consecutive)) == 11 and dice_probability == inclusion_exclusion == Fraction(11, 216)
for sides in range(1, 8):
    for n in range(5):
        explicit = [0] * sides
        for faces in product(range(1, sides + 1), repeat=n):
            explicit[sum(faces) % sides] += 1
        assert residue_counts(n, sides) == explicit
assert ten_dice_counts == [6 ** 9] * 6 and divisible_probability == Fraction(1, 6)
full_sums = {0: 1}
for _ in range(10):
    following = defaultdict(int)
    for total, count in full_sums.items():
        for face in range(1, 7):
            following[total + face] += count
    full_sums = following
assert Fraction(sum(count for total, count in full_sums.items() if total % 6 == 0), 6 ** 10) == divisible_probability
assert median_exact == Fraction(5, 32)
oracle_draws = np.random.default_rng(42).uniform(0, 2, size=(median_trials, 3))
oracle_median = np.count_nonzero(np.count_nonzero(oracle_draws > 1.5, axis=1) >= 2) / median_trials
assert median_estimate == oracle_median
assert monte_carlo(101, batch_size=7) == monte_carlo(101, batch_size=101)
threshold_cases = np.array([[1.5, 1.5, 2], [0, 1.6, 2], [1, 1.5, 1.5], [2, 2, 2]])
assert np.array_equal(np.median(threshold_cases, axis=1) > 1.5, np.count_nonzero(threshold_cases > 1.5, axis=1) >= 2)
def within_six_se(success_rate, exact, n):
    return abs(float(success_rate) - float(exact)) < 6 * sqrt(float(exact * (1 - exact)) / n)
assert within_six_se(median_estimate, median_exact, median_trials)
for n in range(2, 101):
    favourable = sum(right == 2 * left for left, right in combinations(range(1, n + 1), 2))
    assert favourable == n // 2
    assert Fraction(favourable, comb(n, 2)) == Fraction(n // 2, comb(n, 2))
assert len(doubled_pairs) == 50 and len(card_pairs) == 4950 and double_probability == Fraction(1, 99)
for n in range(9):
    for p in [Fraction(0), Fraction(1, 4), Fraction(1, 2), Fraction(1)]:
        pmf = binomial_pmf(n, p)
        assert sum(pmf) == 1 and len(pmf) == n + 1
        mean = sum(k * value for k, value in enumerate(pmf))
        variance = sum((k - mean) ** 2 * value for k, value in enumerate(pmf))
        assert mean == n * p and variance == n * p * (1 - p)
for days in range(1, 5):
    for people in range(6):
        outcomes = list(product(range(days), repeat=people))
        observed = Fraction(sum(len(set(o)) == people for o in outcomes), len(outcomes))
        assert distinct_birthdays(people, days) == observed
assert math.isclose(float(birthday_distinct), 0.293683757280731, rel_tol=1e-12)
assert 1 - distinct_birthdays(23) > Fraction(1, 2)
assert 1 - distinct_birthdays(57) > Fraction(99, 100)
assert always_tails_posterior == Fraction(32, 33)
assert posterior_unfair(0, Fraction(1, 5)) == Fraction(1, 2)
for prior in [Fraction(0), Fraction(1, 3), Fraction(1, 2), Fraction(1)]:
    for q in [Fraction(1, 5), Fraction(1, 2), Fraction(1)]:
        for n in range(7):
            # Enumerate both full joint models and condition on all tails.
            evidence = []
            for likelihood_p, weight in [(q, prior), (Fraction(1, 2), 1 - prior)]:
                evidence.append(sum(weight * math.prod(likelihood_p if x == 'T' else 1 - likelihood_p for x in seq)
                                    for seq in product('HT', repeat=n) if all(x == 'T' for x in seq)))
            if sum(evidence):assert posterior_unfair(n, q, prior) == evidence[0] / sum(evidence)
assert np.allclose(stick_pieces.sum(axis=1), 1) and np.all(stick_pieces >= 0)
assert np.array_equal(stick_successes, np.all(stick_pieces < 0.5, axis=1))
assert np.array_equal(stick_successes, np.all(2 * stick_pieces < stick_pieces.sum(axis=1)[:, None], axis=1))
assert within_six_se(stick_successes.mean(), stick_exact, len(stick_successes))
assert crossing_flags == [False, True, False] and chord_exact == Fraction(1, 3)
def orient(a, b, c):
    return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
points = np.column_stack((np.cos(2 * np.pi * endpoints[:1000]), np.sin(2 * np.pi * endpoints[:1000]))).reshape(1000, 2, 4).transpose(0, 2, 1)
for flags, pts in zip(chord_successes[:1000], points):
    pa, pb, pc, pd = pts
    geometric = orient(pa, pb, pc) * orient(pa, pb, pd) < 0 and orient(pc, pd, pa) * orient(pc, pd, pb) < 0
    assert flags == geometric
assert within_six_se(chord_successes.mean(), chord_exact, len(chord_successes))
assert airplane_probability == 1 - Fraction(17, 20) ** 6
assert math.isclose(float(airplane_probability), 0.622850484375, rel_tol=1e-12)
for n in range(6):
    p = Fraction(3, 20)
    enumerated = sum(math.prod(p if outcome else 1 - p for outcome in outcomes)
                     for outcomes in product([False, True], repeat=n) if any(outcomes))
    assert at_least_one_independent(p, n) == enumerated
for numerator in range(1, 10):
    p = Fraction(numerator, 10)
    assert p * (1 - p) == (1 - p) * p
    assert p * (1 - p) / (2 * p * (1 - p)) == Fraction(1, 2)
for sequence, expected in [('HT', 'H'), ('TH', 'T'), ('HHTTHT', 'H'), ('TTHHTH', 'T')]:
    stream = iter(sequence)
    assert fair_outcome(lambda: next(stream)) == expected
assert len(fair_samples) == 10000 and set(fair_samples) == {'H', 'T'}
assert within_six_se(Fraction(fair_samples.count('H'), len(fair_samples)), Fraction(1, 2), len(fair_samples))
raises(TimeoutError, fair_outcome, lambda: 'H', max_pairs=4)
raises(TimeoutError, fair_outcome, lambda: 'T', max_pairs=4)
raises(ValueError, fair_outcome, lambda: 'invalid')
for function, args in [(monte_carlo, (0,)), (monte_carlo, (-1,)), (monte_carlo, (True,)),
                       (residue_counts, (-1,)), (residue_counts, (2, 0)),
                       (binomial_pmf, (-1, Fraction(1, 2))), (binomial_pmf, (2, 2)),
                       (distinct_birthdays, (-1,)), (distinct_birthdays, (1, 0)),
                       (posterior_unfair, (-1, 1)), (posterior_unfair, (5, -1)),
                       (posterior_unfair, (1, 0, 1)), (broken_stick_trials, (0,)),
                       (at_least_one_independent, (2, 6)), (at_least_one_independent, (0.5, -1)),
                       (chords_cross, ((1, 2), (2, 3)))]:
    raises(ValueError, function, *args)
print('All twelve questions passed exact, enumeration, moment, geometry and edge-case checks.')
print('Original million-trial median simulation matches an independent threshold-count oracle exactly.')
print('Dice/card counts, small birthday spaces, full Bayes joint models and independent interval enumeration agree.')
print('100,000 broken-stick and chord trials pass six-standard-error smoke checks; 1,000 chord intersections match Cartesian geometry.')
print('10,000 extracted fair outcomes pass the seeded smoke check; exact HT/TH symmetry and rejection/timeout paths pass.')


## Original recovered Monte Carlo gist (not executed)

[Source gist](https://gist.github.com/youssefHosni/339b95adbe24b0ad6d7cd8f6dc0d0b40) · retrieved revision `851e5ae22077bbf807c6eac0c48478adaef37f2f`

```python
import numpy as np

def monte_carlo(n):
    count = 0 #to count the number of favourable cases 
    for i in range(n):
        arr = np.random.uniform(0,2,3) # creating 2 numbers randomly under uniform disteibution over [0,2]
        med = np.median(arr) # finding the median of the above 3 numbers 
        if med > 1.5: 
            count+= 1
    return count/n # return the probability
    
# choosing an aritrarily large number for better approximation 
monte_carlo(10**6)
    
```
